In [15]:
import glob
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
result_paths = glob.glob('./Result/analysis_reports_labeled/*')
result_paths = sorted(result_paths, key=lambda x: x.split('/')[-1])

In [4]:
df_snippet_score = pd.DataFrame(columns=['company_name', 'date', 'item_1a_scores', 'item_7_scores'])

for f_p in (result_paths):
    with open(f_p, 'r', encoding='utf-8') as f:
        result = json.load(f)[0]
    company_name = f_p.split('/')[-1].split('_')[0]
    date = f_p.split('/')[-1].split('_')[1].split('.')[0]
    
    item_1a_scores = []
    item_7_scores = []
    
    evidence_list = result['evidence_list']
    for i, evidence in enumerate(evidence_list):
        item_label = evidence.get('item_label', '')
        final_snippet_score = evidence['score_calculation']['final_snippet_score']
        if item_label == 'item_1a':
            item_1a_scores.append(final_snippet_score)
        elif item_label == 'item_7':
            item_7_scores.append(final_snippet_score)
        elif item_label == 'item_1a, item_7':
            item_1a_scores.append(final_snippet_score)
            item_7_scores.append(final_snippet_score)
        else:
            continue
    df_snippet_score.loc[len(df_snippet_score)] = [company_name, date, item_1a_scores, item_7_scores]
    
df_snippet_score['total_scores'] = df_snippet_score['item_1a_scores'] + df_snippet_score['item_7_scores']
# df_snippet_score.to_csv('./Result/llm_result/snippet_scores_list.csv', index=False, encoding='utf-8-sig')
df_snippet_score

,company_name,date,item_1a_scores,item_7_scores,total_scores
0,AAPL,2021-10-29,"[6, 3, 3, 3, 5, 7, 3, 2, 1, 3, 4, 2, 4, 2, 5, ...","[3, 3, 3, 7, 3, 5, 7, 2, 3]","[6, 3, 3, 3, 5, 7, 3, 2, 1, 3, 4, 2, 4, 2, 5, ..."
1,AAPL,2022-10-28,"[8, 3, 2, 3, 7, 9, 3, 0, 5, 5, 3, 7, 4, 3, 6, ...","[3, 2, 3, 4, 3, 6, 6, 2]","[8, 3, 2, 3, 7, 9, 3, 0, 5, 5, 3, 7, 4, 3, 6, ..."
2,AAPL,2023-11-03,"[4, 6]",[],"[4, 6]"
3,AAPL,2024-11-01,"[6, 1]",[],"[6, 1]"
4,ABBV,2021-02-19,"[7, 3, 4, 3, 2, 2, 3, 1]",[3],"[7, 3, 4, 3, 2, 2, 3, 1, 3]"
...,...,...,...,...,...
1723,ZBRA,2024-02-15,"[2, 2, 1, 6, 2, 5, 5, 5, 1]",[1],"[2, 2, 1, 6, 2, 5, 5, 5, 1, 1]"
1724,ZTS,2021-02-16,"[10, 6, 9, 7, 9, 8, 6, 4, 6, 4, 5, 4, 6]","[7, 4, 5, 7, 4, 6]","[10, 6, 9, 7, 9, 8, 6, 4, 6, 4, 5, 4, 6, 7, 4,..."
1725,ZTS,2022-02-15,"[8, 7, 7, 7, 6, 6, 7, 3, 4, 4, 4]","[6, 4]","[8, 7, 7, 7, 6, 6, 7, 3, 4, 4, 4, 6, 4]"
1726,ZTS,2023-02-14,"[9, 8, 5, 5, 6, 7, 1, 4, 3, 1, 1, 2, 3, 1, 2, ...","[8, 5, 9, 3, 1]","[9, 8, 5, 5, 6, 7, 1, 4, 3, 1, 1, 2, 3, 1, 2, ..."


In [30]:
def get_risk_score(item_type='total', cal_func='sum'):
    target_column = item_type + '_scores'
    df_target_score = df_snippet_score[['date', 'company_name', target_column]]
    mask = df_target_score[target_column].map(bool)  # 空[]→False，非空→True
    df_target_score = df_target_score[mask].reset_index(drop=True)

    if cal_func == 'sum':
        df_target_score['risk_score'] = df_target_score[target_column].map(sum)
    elif cal_func == 'max':
        df_target_score['risk_score'] = df_target_score[target_column].map(max)
    elif cal_func == 'min':
        df_target_score['risk_score'] = df_target_score[target_column].map(min)
    elif cal_func == 'mean':
        df_target_score['risk_score'] = df_target_score[target_column].map(np.mean)
    elif cal_func == 'std':
        df_target_score['risk_score'] = df_target_score[target_column].map(np.std)

    df_target_score['date'] = pd.to_datetime(df_target_score['date'], format='%Y-%m-%d')
    df_target_score['year'] = df_target_score['date'].dt.year - 1
    df_target_score['ticker'] = df_target_score['company_name']
    df_target_score = df_target_score[['year', 'ticker', 'risk_score']]
    return df_target_score.dropna()

In [29]:
item_types = ['total', 'item_1a', 'item_7']
cal_funcs = ['sum', 'max', 'min', 'mean', 'std']

for i_t in item_types:
    for c_f in cal_funcs:
        save_path = f'./Data/Fama_French/risk_scores_{i_t}_{c_f}.csv'
        df_target_score = get_risk_score(i_t, c_f)
        df_target_score.to_csv(save_path, index=False)